# Lab 5 — Linear Regression Fit (Zomato Ratings)

**Day 02 · Python for Data Science · Cisco AI/ML Training**

---

## Learning objectives

1. Define **features** ($X$) and **target** ($y$) for a regression problem.
2. Fit **Ordinary Least Squares (OLS)** with `sklearn.linear_model.LinearRegression`.
3. Interpret **intercept** and **coefficients** in the prediction equation.
4. Generate and compare **predicted vs actual** ratings.

> **Checkpoints:** **500** training rows · intercept ≈ **3.72** · two feature coefficients printed

**Companion script:** `../scripts/lab05_linear_regression_fit.py`


## The linear regression model

For multiple features, OLS estimates:

$$
\hat{y} = eta_0 + eta_1 x_1 + eta_2 x_2 + \cdots + eta_p x_p
$$

| Symbol | Meaning |
|--------|---------|
| $\hat{y}$ | Predicted `aggregate_rating` |
| $eta_0$ | Intercept (`model.intercept_`) |
| $eta_j$ | Coefficient for feature $j$ (`model.coef_`) |
| $x_j$ | Input values (votes, cost, …) |

**Goal:** Choose $eta$ values that minimize the sum of squared residuals on training data.


---

## 1. Load data and define X, y

We predict **`aggregate_rating`** from engagement (`votes`) and price (`average_cost_for_two`).


In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-02":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "zomato" / "zomato_restaurants.csv").is_file():
            GH_ROOT = parent
            break

ZOMATO_CSV = GH_ROOT / "data" / "zomato" / "zomato_restaurants.csv"
df = pd.read_csv(ZOMATO_CSV)

FEATURES = ["votes", "average_cost_for_two"]
TARGET = "aggregate_rating"

X = df[FEATURES]
y = df[TARGET]

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(X.head(3))


### Why these features?

- **`votes`** — customer engagement proxy.
- **`average_cost_for_two`** — price positioning.

In real Zomato analytics you might add cuisine, city, or binary flags. Lab 6 evaluates how well this simple model generalizes.


---

## 2. Visualize relationships (optional but recommended)

Scatter plots reveal whether a **linear** model is plausible.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sns.scatterplot(data=df, x="votes", y=TARGET, ax=axes[0], alpha=0.5, s=20)
axes[0].set_title("Rating vs votes")

sns.scatterplot(data=df, x="average_cost_for_two", y=TARGET, ax=axes[1], alpha=0.5, s=20, color="coral")
axes[1].set_title("Rating vs average cost for two")

plt.tight_layout()
plt.show()


On this **synthetic** classroom dataset the cloud may look weakly related — that is intentional. The lab teaches **mechanics**; Lab 6 quantifies fit quality with metrics.


---

## 3. Instantiate and fit the model

`LinearRegression()` uses OLS by default. `.fit(X, y)` learns $eta_0, eta_1, \ldots$ from all **500** rows.


In [ ]:
model = LinearRegression()
model.fit(X, y)

print("Lab 5 — Linear regression fit")
print(f"training rows: {len(df)}")
print(f"intercept (beta_0): {model.intercept_:.4f}")
print(f"coefficients: {dict(zip(FEATURES, model.coef_.round(6)))}")

assert len(df) == 500


---

## 4. Manual prediction for one row

Verify sklearn math by hand for the first restaurant:

$$
\hat{y} = eta_0 + eta_1 \cdot 	ext{votes} + eta_2 \cdot 	ext{cost}
$$


In [ ]:
row = X.iloc[0]
manual_pred = model.intercept_ + model.coef_[0] * row["votes"] + model.coef_[1] * row["average_cost_for_two"]
sklearn_pred = model.predict(X.head(1))[0]

print("Row 0 features:", row.to_dict())
print(f"Manual prediction:   {manual_pred:.4f}")
print(f"sklearn prediction:  {sklearn_pred:.4f}")
print(f"Actual rating:       {y.iloc[0]:.1f}")


---

## 5. Batch predictions — first three rows


In [ ]:
from IPython.display import display
predictions = model.predict(X.head(3))
actual = y.head(3).tolist()

comparison = pd.DataFrame({
    "votes": X.head(3)["votes"].values,
    "cost": X.head(3)["average_cost_for_two"].values,
    "actual_rating": actual,
    "predicted_rating": predictions.round(2),
})
display(comparison)

print("sample predictions (first 3):", predictions.round(2))
print("actual ratings (first 3):", actual)


**Note:** When coefficients are near zero, predictions cluster near the intercept (~ mean rating). That signals weak linear signal — not a coding error.


---

## 6. Extension — add `online_order` feature

Encode Yes/No as 1/0 and refit. Compare coefficients.


In [ ]:
df_ext = df.copy()
df_ext["online_order_num"] = (df_ext["online_order"] == "Yes").astype(int)

FEATURES_EXT = FEATURES + ["online_order_num"]
X_ext = df_ext[FEATURES_EXT]

model_ext = LinearRegression()
model_ext.fit(X_ext, y)

print("Extended model coefficients:")
for name, coef in zip(FEATURES_EXT, model_ext.coef_):
    print(f"  {name}: {coef:.6f}")
print(f"intercept: {model_ext.intercept_:.4f}")


---

## 7. Residual preview (bridge to Lab 6)

**Residual** = actual − predicted. Large residuals → model misses those points.


In [ ]:
y_pred_all = model.predict(X)
residuals = y - y_pred_all

fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(residuals, bins=20, kde=True, ax=ax)
ax.set_title("Residual distribution (in-sample)")
ax.set_xlabel("actual - predicted")
plt.tight_layout()
plt.show()

print(f"Mean residual (should ~0): {residuals.mean():.4f}")


---

## 8. Checkpoint summary


In [ ]:
print("=" * 50)
print("CHECKPOINT SUMMARY")
print("=" * 50)
print(f"training rows: {len(df)}")
print(f"intercept: {model.intercept_:.4f}")
print(f"coefficients [votes, cost]: {model.coef_.round(4)}")

assert len(df) == 500
assert len(model.coef_) == 2
print("\n✓ Checkpoint assertions passed")


## Concept — gradient descent (lecture link)

<!-- cisco-enrich-2026-06 -->

`LinearRegression` solves OLS **analytically** (closed form). For huge datasets or neural nets, **gradient descent** iteratively adjusts weights to minimize loss. Same goal — different compute path.

| Method | When used |
|--------|----------|
| OLS / normal equation | Small/medium tabular (this lab) |
| Gradient descent | Deep learning, online learning |

## Extension — compare votes-only vs two-feature model

In [ ]:
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

y = df["aggregate_rating"]
X1 = df[["votes"]]
X2 = df[["votes", "average_cost_for_two"]]
for name, X in [("votes only", X1), ("votes + cost", X2)]:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    m = LinearRegression().fit(X_tr, y_tr)
    print(f"{name:14s} R2 test = {r2_score(y_te, m.predict(X_te)):.4f}")


## Linear vs non-linear relationships (course topic)

<!-- cisco-topic-coverage -->

**Linear regression** assumes a straight-line relationship between features and target. Real data often curves — e.g. ratings plateau at high vote counts.

| Model family | When |
|--------------|------|
| Linear (this lab) | Interpretable baseline, fast |
| Polynomial / trees | Non-linear patterns (Days 5–6) |


In [ ]:
# optional: polynomial feature to capture curvature
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

X = df[["votes"]]
y = df["aggregate_rating"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
lin = LinearRegression().fit(X_tr, y_tr)
poly = make_pipeline(PolynomialFeatures(degree=2), LinearRegression()).fit(X_tr, y_tr)
print(f"linear R2:      {r2_score(y_te, lin.predict(X_te)):.4f}")
print(f"polynomial R2:  {r2_score(y_te, poly.predict(X_te)):.4f}")


---

## Reflection questions

1. What does the intercept represent when all features are zero? Is that scenario realistic for votes/cost?
2. Why might we scale features before regression? *(Preview: Day 3 `StandardScaler`)*
3. What metrics will quantify goodness-of-fit? *(Lab 6 — R², RMSE)*

**Previous:** [Lab 4 — Seaborn plots](lab04_seaborn_plots.ipynb)  
**Next:** [Lab 6 — LR evaluation metrics](lab06_lr_evaluation_metrics.ipynb)
